In [3]:
import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook

OBJECTIVES = [
    ('max', 'Max Bisection'),
    ('min', 'Min Bisection'),
    ('fair_max', 'Fair Max Bisection'),
    ('fair_min', 'Fair Min Bisection'),
]
DATASETS = {'enron': ('Email-Enron', 9), 'congress': ('Congress-bills', 10)}





In [4]:


def load_results(path, depth):
    book = load_workbook(path, read_only=True, data_only=True)
    results = {}
    for sheet in book.worksheets:
        name = sheet.title.strip()  # Handles the leading space in ' max_enron'
        if not any(name == f'{obj}_{dataset}' for dataset in DATASETS for obj, _ in OBJECTIVES):
            continue
        # These sheets have at most 13 columns and 41 populated rows.
        cells = sheet.iter_rows(values_only=True)
        headers = [str(h).strip() if h is not None else '' for h in next(cells)]
        for values in cells:
            if not values[0]:
                continue
            row = dict(zip(headers, values))
            try:
                if int(float(row['p'])) != depth or str(row['runtype']).strip().lower() != 'aon':
                    continue
                for key in ('n_vertices', 'n_hyperedges', 'initial_objective',
                            'final_objective', 'optimal', 'ratio'):
                    row[key] = float(row[key])
            except (ValueError, TypeError, KeyError):
                print(f'Skipping incomplete data in sheet {sheet.title}: {row.get("instance")}')
                continue
            results[(name, str(row['instance']))] = row
    return results




In [5]:


def load_results(path, depth):
    book = load_workbook(path, read_only=True, data_only=True)
    results = {}
    for sheet in book.worksheets:
        name = sheet.title.strip()  # Handles the leading space in ' max_enron'
        if not any(name == f'{obj}_{dataset}' for dataset in DATASETS for obj, _ in OBJECTIVES):
            continue
        # These sheets have at most 13 columns and 41 populated rows.
        cells = sheet.iter_rows(values_only=True)
        headers = [str(h).strip() if h is not None else '' for h in next(cells)]
        for values in cells:
            if not values[0]:
                continue
            row = dict(zip(headers, values))
            try:
                if int(float(row['p'])) != depth or str(row['runtype']).strip().lower() != 'aon':
                    continue
                for key in ('n_vertices', 'n_hyperedges', 'initial_objective',
                            'final_objective', 'optimal', 'ratio'):
                    row[key] = float(row[key])
            except (ValueError, TypeError, KeyError):
                print(f'Skipping incomplete data in sheet {sheet.title}: {row.get("instance")}')
                continue
            results[(name, str(row['instance']))] = row
    return results



In [6]:
def get_rows(data, dataset, objective, n):
    return [data.get((f'{objective}_{dataset}',
                      f'{"email-Enron" if dataset == "enron" else "congress-bills"}_{i:02d}'
                      if dataset == 'congress' else f'email-Enron_{i}'))
            for i in range(1, n + 1)]




In [8]:
def make_heatmap(data, dataset, depth, output):
    label, count = DATASETS[dataset]
    scores = np.full((len(OBJECTIVES), count), np.nan)
    for j, (obj, _) in enumerate(OBJECTIVES):
        for i, row in enumerate(get_rows(data, dataset, obj, count)):
            if row is not None:
                scores[j, i] = row['ratio'] * 100

    fig, ax = plt.subplots(figsize=(max(8, count * 0.85), 3.8), constrained_layout=True)
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('#eeeeee')
    # Set a common, easily interpreted scale across both datasets.
    image = ax.imshow(np.ma.masked_invalid(scores), cmap=cmap, vmin=0, vmax=100,
                      aspect='auto')
    ax.set_xticks(np.arange(count), [str(i) for i in range(1, count + 1)])
    ax.set_yticks(np.arange(len(OBJECTIVES)), [name for _, name in OBJECTIVES])
    ax.set_xlabel('Hypergraph instance')
    ax.set_title(f'{label}: QAOA performance (p={depth})')
    for j in range(scores.shape[0]):
        for i in range(scores.shape[1]):
            value = scores[j, i]
            if not np.isfinite(value):
                text = 'N/A'
                color = 'black'
            else:
                text = f'{value:.1f}'
                color = 'white' if value < 45 else 'black'
            ax.text(i, j, text, ha='center', va='center', fontsize=8, color=color)
    colorbar = fig.colorbar(image, ax=ax, shrink=0.85, pad=0.015)
    colorbar.set_label('Performance ratio (%)')
    save(fig, output / f'heatmap_{dataset}_p{depth}')



In [10]:
def make_objective_bars(data, dataset, depth, output):
    label, count = DATASETS[dataset]
    fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), constrained_layout=True)
    x = np.arange(count)
    for ax, (obj, obj_label) in zip(axes.flat, OBJECTIVES):
        rows = get_rows(data, dataset, obj, count)
        exact = np.array([r['optimal'] if r else np.nan for r in rows])
        qaoa = np.array([r['final_objective'] if r else np.nan for r in rows])
        ax.bar(x - 0.18, exact, 0.36, label='Gurobi optimum')
        ax.bar(x + 0.18, qaoa, 0.36, label=f'QAOA p={depth}')
        ax.set_title(obj_label)
        ax.set_xticks(x, [str(i) for i in range(1, count + 1)])
        ax.set_xlabel('Hypergraph instance')
        ax.set_ylabel('Objective value')
        ax.grid(axis='y', alpha=0.2)
        if not np.isfinite(qaoa).any():
            ax.text(0.5, 0.5, f'No p={depth} data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=12)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside upper center', ncol=2)
    fig.suptitle(f'{label}: QAOA vs. Gurobi (p={depth})', fontsize=14)
    save(fig, output / f'objectives_{dataset}_p{depth}')



In [11]:
def make_size_scatter(data, depth, output):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
    for ax, (obj, name) in zip(axes.flat, OBJECTIVES):
        for dataset, (dataset_label, count) in DATASETS.items():
            rows = [r for r in get_rows(data, dataset, obj, count) if r]
            if not rows:
                continue
            x = [r['n_vertices'] for r in rows]
            y = [r['ratio'] * 100 for r in rows]
            ax.scatter(x, y, label=dataset_label, alpha=0.8, s=55)
        ax.set_title(name)
        ax.set_xlabel('Number of vertices')
        ax.set_ylabel('Performance ratio (%)')
        ax.set_ylim(0, 103)
        ax.grid(alpha=0.2)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='outside upper center', ncol=2)
    fig.suptitle(f'Performance versus graph size (p={depth})', fontsize=14)
    save(fig, output / f'size_vs_performance_p{depth}')



In [12]:
def save(fig, stem):
    fig.savefig(stem.with_suffix('.png'), dpi=300, bbox_inches='tight')
    fig.savefig(stem.with_suffix('.pdf'), bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {stem}.png and .pdf')



In [18]:
from pathlib import Path

excel_path = "results_combined.xlsx"  # Update path if needed
output = Path("figures")
output.mkdir(parents=True, exist_ok=True)

depth = 3

data = load_results(excel_path, depth)

print(f"Loaded {len(data)} experiment results")

Loaded 76 experiment results


In [19]:
make_heatmap(data, "enron", depth, output)

Saved figures/heatmap_enron_p3.png and .pdf


In [20]:
make_heatmap(data, "congress", depth, output)

Saved figures/heatmap_congress_p3.png and .pdf


In [21]:
make_objective_bars(data, "enron", depth, output)

Saved figures/objectives_enron_p3.png and .pdf


In [22]:
import matplotlib.pyplot as plt
import numpy as np

def make_average_ratio_heatmap(data, depth, output):
    dataset_names = list(DATASETS.keys())
    matrix = np.full((len(OBJECTIVES), len(dataset_names)), np.nan)

    for i, (obj, obj_label) in enumerate(OBJECTIVES):
        for j, dataset in enumerate(dataset_names):
            _, count = DATASETS[dataset]
            rows = [r for r in get_rows(data, dataset, obj, count) if r is not None]
            if rows:
                ratios = [r['ratio'] * 100 for r in rows]
                matrix[i, j] = np.nanmean(ratios)

    fig, ax = plt.subplots(figsize=(6, 4.5), constrained_layout=True)

    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("#eeeeee")

    im = ax.imshow(np.ma.masked_invalid(matrix), cmap=cmap, vmin=0, vmax=100, aspect="auto")

    ax.set_xticks(np.arange(len(dataset_names)), ["Email-Enron", "Congress-bills"])
    ax.set_yticks(np.arange(len(OBJECTIVES)), [name for _, name in OBJECTIVES])

    ax.set_title(f"Average QAOA performance ratio by dataset (p={depth})")
    ax.set_xlabel("Dataset")

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            if np.isfinite(val):
                text = f"{val:.1f}"
                color = "white" if val < 45 else "black"
            else:
                text = "N/A"
                color = "black"
            ax.text(j, i, text, ha="center", va="center", fontsize=10, color=color)

    cbar = fig.colorbar(im, ax=ax, shrink=0.9, pad=0.02)
    cbar.set_label("Average performance ratio (%)")

    save(fig, output / f"avg_ratio_heatmap_p{depth}")

In [27]:
data = load_results(excel_path, 6)
make_average_ratio_heatmap(data, 6, output)

Saved figures/avg_ratio_heatmap_p6.png and .pdf
